# Hybrid ML + DL model

**Owner:** Kapish Rohilla. Phase 2 (deep learning), step 4 of 5.

## Purpose

Phase 1 tree models and Phase 2 sequence models see the engine history in different ways. The Random Forest uses 57 hand-made features over the last 10 cycles; the LSTM, GRU, 1D-CNN and Transformer read a raw 30-cycle window. This notebook tests whether a **hybrid** that combines the two is better than either part, and whether any gain is worth the added complexity compared with a plain average.

Phase 1 weaknesses this targets:
1. every Phase 1 model underpredicts engines with true RUL above 200 cycles;
2. the 10-cycle rolling features only see recent behaviour;
3. RMSE is much larger than MAE, so a few large errors dominate.

## Method

### Design (written before any experiment)

**Hybrid A (stacking).** A deep model $f_\theta$ maps the 30-cycle window $x_{t-29:t}$ of an engine to a RUL estimate

$$d_t = f_\theta(x_{t-29:t}).$$

Phase 1's rolling features $\phi(x)_t \in \mathbb{R}^{57}$ (cycle, 14 sensors, and the 10-cycle mean, standard deviation and slope of each sensor) are joined with $d_t$, and a tree regressor $g$ (Random Forest or Gradient Boosting, Phase 1 hyperparameters, not retuned) predicts

$$\hat y_t = g\big([\phi(x)_t,\ d_t]\big).$$

**Hybrid B (average, no fitted weights).** $\hat y_t = \tfrac12\big(d^{\text{LSTM}}_t + \hat y^{\text{RF}}_t\big)$, the mean of the best DL model (`lstm_v1`) and the best Phase 1 model (Random Forest). If B is as good as A, the stacking is not justified.

**Hybrid C (residual).** The tree predicts the DL error $r_t = y_t - d_t$ from the same features, and $\hat y_t = d_t + g([\phi(x)_t, d_t])$.

**Why a scalar $d$ and not the embedding.** Each trained network has its own embedding coordinates, so the 5 fold networks below produce embeddings that cannot be mixed in one tree column. A RUL estimate in cycles means the same thing for every network. A bottleneck-embedding variant was left out for this reason and for time.

### Hypothesis

The sequence model summarises a longer history (30 cycles) than the rolling features (10 cycles), and the tree mixes it nonlinearly with the engineered features and can correct the DL model where it is systematically wrong. So Hybrid A should beat both of its parts on the validation engines, especially for high true RUL, and should beat the plain average B.

### Leakage control: out-of-fold predictions by engine

If the tree were trained on $d$ from an LSTM that was itself trained on the same 80 engines, $d$ would be far more accurate on those rows than on new engines, the tree would learn to trust it too much, and validation error would rise. So the training values of $d$ are **out-of-fold**:

1. The 80 training engines are split into $K=5$ folds **by engine** (`hybrid.engine_folds`, seeded NumPy permutation).
2. For fold $k$, a fresh LSTM with the `config_lstm.json` settings is trained on the other 64 engines: 56 for gradient steps and 8 for early stopping (`hybrid.split_fit_stop`). The scaler is fitted on the 56 engines. Fold $k$ and the 20 validation engines are never seen.
3. The model predicts the 16 engines of fold $k$. Every training row therefore gets a $d$ from a model that never saw its engine.
4. **Validation and test $d$** = mean of the 5 fold models. We chose this over the saved `lstm_v1` because (a) the fold models never saw the validation engines, while `lstm_v1` used them for early stopping, so selecting on them would be optimistic; (b) each fold model was trained on 64 engines, like the models behind the out-of-fold training values, so training and inference $d$ come from the same kind of model; (c) the whole procedure can be repeated per seed. The same choice is used for validation and test. Remaining mismatch: a mean of 5 models is smoother and slightly more accurate than one out-of-fold model.

`tests/test_hybrid.py` checks fold assignment by engine, that no fold model saw its held-out engines or any validation engine, that test data (no `rul`) is refused, and that all joins are by `(engine_id, cycle)`.

### Variants and selection rule (fixed before running)

| Variant | Features | Tree | Target |
| --- | --- | --- | --- |
| A-RF | rolling + $d_{\text{LSTM}}$ | Random Forest | RUL |
| A-GB | rolling + $d_{\text{LSTM}}$ | Gradient Boosting | RUL |
| A-RF-2DL | rolling + $d_{\text{LSTM}}$ + $d_{\text{GRU}}$ | Random Forest | RUL |
| C-RF | rolling + $d_{\text{LSTM}}$ | Random Forest | $y - d_{\text{LSTM}}$ |
| B | – | – | mean of `lstm_v1` and Phase 1 RF |

- The DL model is the one with the lowest `best_val_window_mae` in the four `config_*.json` files (the Yogesh rule); the second-best recurrent model supplies $d_{\text{GRU}}$.
- **Selection:** lowest MAE on all windows of the 20 validation engines; ties within 0.05 cycles go to the simpler variant. The 100 fixed points are reported once, after selection, and are not used to choose.
- Training target is the true, uncapped RUL; evaluation always uses the true RUL.
- The chosen variant is repeated with seeds 42, 43, 44 (folds, LSTMs and tree); saved predictions are seed 42.
- **Runtime limits:** fold LSTMs use the `config_lstm.json` budget (at most 60 epochs, patience 8) on CPU; the GRU second model is trained for seed 42 only. No test metric is computed here.

### Setup

Imports, paths, and the random seed. Torch runs on CPU, which is deterministic and as fast as MPS for these small models (see `02_yogesh` handoff).

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve().parents[1]  # run from notebooks/phase2
sys.path.insert(0, str(ROOT / "src"))

import hybrid
import train
from evaluation import VAL_ENGINES, split_train_val, test_points, validation_points
from hybrid import KEYS
from load_data import load_test, load_train

RESULTS = ROOT / "results" / "phase2"
FIGURES = RESULTS / "figures"
SEED = 42
SEEDS = [42, 43, 44]
START = time.perf_counter()
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 11})

### Data

The 80/20 engine split from Phase 1, the 100 fixed validation points and the 100 test points (last observed cycle of each test engine). Rows are kept as raw data; rolling features and windows are built per engine inside `src/hybrid.py`.

In [ ]:
train_df, val_df = split_train_val(load_train())
test_df = load_test()
val_pts = validation_points(val_df)  # 100 rows, fixed order
test_pts = test_points(test_df)[KEYS].reset_index(drop=True)  # engine 1..100

y_train = train_df.sort_values(KEYS)["rul"].to_numpy(dtype=float)  # same order as the hybrid tables
y_val_win = val_df.sort_values(KEYS)["rul"].to_numpy(dtype=float)
y_val_pts = val_pts["rul"].to_numpy(dtype=float)

assert set(train_df.engine_id).isdisjoint(VAL_ENGINES) and len(val_pts) == len(test_pts) == 100
print(f"training engines {train_df.engine_id.nunique()}, rows {len(train_df):,}")
print(f"validation engines {val_df.engine_id.nunique()}, rows {len(val_df):,}; test engines {test_df.engine_id.nunique()}")

### Check: saved DL models reproduce their reported scores

Before any DL output is used, each saved model from steps 2 and 3 is reloaded and scored on the 100 points. Its MAE must match its `config_*.json` value. The model with the lowest validation-window MAE (selection rule) becomes the hybrid's DL component.

In [ ]:
def load_config(kind: str) -> dict:
    """Read config_<kind>.json and return it with constructor arguments under 'kwargs'."""
    cfg = json.loads((RESULTS / f"config_{kind}.json").read_text())
    if "kwargs" not in cfg:  # the recurrent configs store them at the top level
        cfg["kwargs"] = {k: cfg[k] for k in ("hidden", "layers", "dropout")}
    return cfg


configs = {kind: load_config(kind) for kind in ["lstm", "gru", "cnn1d", "transformer"]}
check_rows = []
for kind, cfg in configs.items():
    model = train.load_model(kind, RESULTS / "models" / f"{kind}_v1.pt", **cfg["kwargs"])
    pred = hybrid.dl_predict(model, hybrid.sequences.fit_scaler(train_df), val_df, cfg["window"])
    pred = val_pts[KEYS].merge(pred, on=KEYS, how="left")
    check_rows.append({
        "model": cfg["model"],
        "selection_window_mae": cfg["best_val_window_mae"],
        "reported_mae": cfg["val_points"]["mae"],
        "reloaded_mae": hybrid.point_metrics(y_val_pts, pred["d"])["mae"],
    })
reload_check = pd.DataFrame(check_rows).sort_values("selection_window_mae")
assert np.allclose(reload_check.reported_mae, reload_check.reloaded_mae, atol=0.01)
DL_KIND, DL2_KIND = "lstm", "gru"
assert reload_check.model.iloc[0] == f"{DL_KIND}_v1"  # best by the selection rule
reload_check.round(3)

### Check: Phase 1 Random Forest is reproduced

The rolling-features-only tree from `hybrid.make_tree` must give the same predictions as Phase 1 (`results/phase1/val_predictions_trees.csv`). This confirms the tree settings were copied correctly. Its predictions are also the tree half of Hybrid B.

In [ ]:
X_roll_train = hybrid.build_hybrid_features(train_df, None, groups=["rolling"])
rf_phase1 = hybrid.fit_hybrid(X_roll_train, y_train, tree="random_forest", seed=SEED)
rf_val_pts = hybrid.predict_hybrid(
    rf_phase1, hybrid.build_hybrid_features(val_df, None, ["rolling"], points=val_pts)
)
phase1_trees = pd.read_csv(ROOT / "results" / "phase1" / "val_predictions_trees.csv")
assert phase1_trees[KEYS].equals(val_pts[KEYS])
print("max |difference| to Phase 1 RF:", np.abs(rf_val_pts - phase1_trees.random_forest).max())
print("Phase 1 RF MAE:", round(hybrid.point_metrics(y_val_pts, rf_val_pts)["mae"], 2))